## Example 1: mercantile functionality test 1

In [ ]:
import mercantile
import math

# Example 1: Basic tile operations and coordinate transformations
print("mercantile module imported successfully")

try:
    print("Example 1 - Basic tile operations and coordinate transformations")
    
    # Sample locations around the world
    locations = [
        ("New York City", -74.006, 40.7128),
        ("London", -0.1276, 51.5074),
        ("Tokyo", 139.6503, 35.6762),
        ("Sydney", 151.2093, -33.8688),
        ("São Paulo", -46.6333, -23.5505)
    ]
    
    print("Converting coordinates to tiles at different zoom levels:")
    print("=" * 60)
    
    zoom_levels = [10, 15, 18]  # Different levels of detail
    
    for city, lon, lat in locations:
        print(f"\n🏙️  {city} ({lat:.4f}, {lon:.4f})")
        
        for zoom in zoom_levels:
            # Convert lat/lon to tile coordinates
            tile = mercantile.tile(lon, lat, zoom)
            
            print(f"  Zoom {zoom:2d}: Tile({tile.x}, {tile.y}, {tile.z})")
            
            # Get the bounding box of this tile
            bbox = mercantile.bounds(tile)
            
            print(f"           Bounds: [{bbox.west:.6f}, {bbox.south:.6f}, {bbox.east:.6f}, {bbox.north:.6f}]")
            
            # Calculate tile size in meters (approximate)
            # At equator: circumference / (2^zoom) tiles
            earth_circumference = 40075017  # meters
            tile_size_meters = earth_circumference / (2 ** zoom)
            print(f"           Approx tile size: {tile_size_meters:.0f}m x {tile_size_meters:.0f}m")
    
    # Demonstrate coordinate round-trip accuracy
    print(f"\nCoordinate Round-trip Accuracy Test:")
    print("-" * 40)
    
    test_location = locations[0]  # New York City
    city, original_lon, original_lat = test_location
    
    for zoom in [12, 16, 20]:
        # Original -> Tile -> Bounds -> Center
        tile = mercantile.tile(original_lon, original_lat, zoom)
        bbox = mercantile.bounds(tile)
        
        # Calculate center of bounding box
        center_lon = (bbox.west + bbox.east) / 2
        center_lat = (bbox.south + bbox.north) / 2
        
        # Calculate error
        lon_error = abs(center_lon - original_lon)
        lat_error = abs(center_lat - original_lat)
        
        print(f"Zoom {zoom}: Error = ({lon_error:.8f}°, {lat_error:.8f}°)")
        print(f"         Distance error ≈ {lon_error * 111320:.2f}m, {lat_error * 110540:.2f}m")
    
    print("mercantile example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mercantile example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: mercantile functionality test 2

In [ ]:
import mercantile

# Example 2: Bounding box operations and tile ranges
print("mercantile bounding box operations and tile ranges")

try:
    print("Example 2 - Bounding box operations and tile ranges")
    
    # Define geographic regions of interest
    regions = {
        "Manhattan": {
            "name": "Manhattan, NYC",
            "bbox": [-74.0479, 40.6829, -73.9067, 40.8820]  # [west, south, east, north]
        },
        "Central_London": {
            "name": "Central London",
            "bbox": [-0.1967, 51.4700, -0.0505, 51.5426]
        },
        "Tokyo_Center": {
            "name": "Tokyo Center",
            "bbox": [139.5538, 35.5493, 139.7922, 35.7341]
        }
    }
    
    print("Analyzing tile requirements for different regions:")
    print("=" * 55)
    
    zoom_levels = [12, 15, 18]
    
    for region_key, region_info in regions.items():
        name = region_info["name"]
        west, south, east, north = region_info["bbox"]
        
        print(f"\n🗺️  Region: {name}")
        print(f"   Bounding Box: [{west:.4f}, {south:.4f}, {east:.4f}, {north:.4f}]")
        
        for zoom in zoom_levels:
            # Get all tiles that intersect with this bounding box
            tiles = list(mercantile.tiles(west, south, east, north, zoom))
            
            # Calculate tile grid dimensions
            x_coords = [tile.x for tile in tiles]
            y_coords = [tile.y for tile in tiles]
            
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)
            
            x_range = x_max - x_min + 1
            y_range = y_max - y_min + 1
            
            print(f"   Zoom {zoom:2d}: {len(tiles):4d} tiles ({x_range} x {y_range} grid)")
            print(f"            X: {x_min} to {x_max}, Y: {y_min} to {y_max}")
            
            # Calculate total area covered by tiles
            first_tile_bbox = mercantile.bounds(tiles[0])
            tile_width = first_tile_bbox.east - first_tile_bbox.west
            tile_height = first_tile_bbox.north - first_tile_bbox.south
            
            total_tile_area = len(tiles) * tile_width * tile_height
            
            # Original bounding box area
            original_area = (east - west) * (north - south)
            coverage_efficiency = (original_area / total_tile_area) * 100
            
            print(f"            Coverage efficiency: {coverage_efficiency:.1f}%")
    
    # Demonstrate parent/child tile relationships
    print(f"\nTile Hierarchy Relationships:")
    print("=" * 35)
    
    # Start with a tile at zoom level 14
    base_tile = mercantile.Tile(x=4823, y=6160, z=14)  # Manhattan area
    print(f"Base tile: {base_tile}")
    
    # Get parent tile (zoom out)
    parent_tile = mercantile.parent(base_tile)
    print(f"Parent tile (z-1): {parent_tile}")
    
    # Get children tiles (zoom in)
    children_tiles = list(mercantile.children(base_tile))
    print(f"Children tiles (z+1): {len(children_tiles)} tiles")
    for i, child in enumerate(children_tiles):
        print(f"  Child {i+1}: {child}")
    
    # Verify parent-child relationships
    print(f"\nRelationship Verification:")
    for child in children_tiles:
        child_parent = mercantile.parent(child)
        if child_parent == base_tile:
            print(f"  ✓ {child} -> {child_parent}")
        else:
            print(f"  ✗ {child} -> {child_parent} (expected {base_tile})")
    
    # Calculate tiles needed for multi-zoom pyramid
    print(f"\nPyramid Analysis for Manhattan:")
    print("-" * 30)
    
    manhattan_bbox = regions["Manhattan"]["bbox"]
    west, south, east, north = manhattan_bbox
    
    total_tiles = 0
    for zoom in range(10, 19):  # Zoom levels 10-18
        tiles = list(mercantile.tiles(west, south, east, north, zoom))
        total_tiles += len(tiles)
        print(f"Zoom {zoom:2d}: {len(tiles):5d} tiles")
    
    print(f"Total tiles needed: {total_tiles:,}")
    print(f"Storage estimate (256x256 PNG): {total_tiles * 15:.1f} KB average")
    
    print("mercantile example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mercantile example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: mercantile functionality test 3

In [ ]:
import mercantile
import math

# Example 3: Projection transformations and coordinate systems
print("mercantile projection transformations and coordinate systems")

try:
    print("Example 3 - Projection transformations and coordinate systems")
    
    # Sample coordinates in different coordinate systems
    locations = [
        ("Equator & Prime Meridian", 0.0, 0.0),
        ("North Pole", 0.0, 89.9),  # Very close to pole
        ("South Pole", 0.0, -89.9),
        ("International Date Line", 180.0, 0.0),
        ("Extreme West", -179.9, 45.0),
        ("Extreme East", 179.9, 45.0)
    ]
    
    print("Geographic to Web Mercator Transformations:")
    print("=" * 50)
    
    for name, lon, lat in locations:
        print(f"\n📍 {name}")
        print(f"   Geographic (WGS84): ({lat:.3f}°, {lon:.3f}°)")
        
        # Convert to Web Mercator coordinates
        try:
            x, y = mercantile.xy(lon, lat)
            print(f"   Web Mercator: ({x:.2f}, {y:.2f}) meters")
            
            # Convert back to lat/lon
            lon_back, lat_back = mercantile.lnglat(x, y)
            print(f"   Round-trip: ({lat_back:.6f}°, {lon_back:.6f}°)")
            
            # Calculate transformation errors
            lat_error = abs(lat - lat_back)
            lon_error = abs(lon - lon_back)
            print(f"   Error: ({lat_error:.8f}°, {lon_error:.8f}°)")
            
        except Exception as e:
            print(f"   Transformation error: {str(e)}")
            raise
    
    # Analyze projection distortion at different latitudes
    print(f"\nProjection Distortion Analysis:")
    print("=" * 35)
    
    reference_longitude = 0.0  # Prime Meridian
    latitudes = [0, 15, 30, 45, 60, 75, 85]  # Different latitudes
    
    print("Latitude | Scale Factor | Distortion")
    print("---------|--------------|------------")
    
    for lat in latitudes:
        # Calculate scale factor (how much distances are stretched)
        # Web Mercator scale factor = 1 / cos(latitude in radians)
        lat_rad = math.radians(lat)
        scale_factor = 1.0 / math.cos(lat_rad) if lat < 85 else float('inf')
        
        distortion_percent = (scale_factor - 1.0) * 100
        
        if scale_factor < 10:  # Only show reasonable values
            print(f" {lat:7.0f}° | {scale_factor:10.3f} | {distortion_percent:+8.1f}%")
        else:
            print(f" {lat:7.0f}° | {'    inf':>10} | {'   inf':>8}%")
    
    # Demonstrate tile coordinate precision at different zooms
    print(f"\nTile Coordinate Precision Analysis:")
    print("=" * 40)
    
    test_location = (0.0, 0.0)  # Null Island (Equator & Prime Meridian)
    lon, lat = test_location
    
    print(f"Location: ({lat:.6f}, {lon:.6f})")
    print("Zoom | Tile Position | Sub-tile Precision")
    print("-----|---------------|-------------------")
    
    for zoom in [5, 10, 15, 20]:
        # Get tile coordinates
        tile = mercantile.tile(lon, lat, zoom)
        
        # Calculate sub-tile position (0.0 to 1.0)
        bbox = mercantile.bounds(tile)
        
        x_fraction = (lon - bbox.west) / (bbox.east - bbox.west)
        y_fraction = (lat - bbox.south) / (bbox.north - bbox.south)
        
        # Calculate precision in degrees
        tile_width = bbox.east - bbox.west
        tile_height = bbox.north - bbox.south
        
        print(f" {zoom:3d}  | ({tile.x}, {tile.y}) | ({x_fraction:.4f}, {y_fraction:.4f})")
        print(f"      | Tile size: {tile_width:.6f}° x {tile_height:.6f}°")
    
    # Edge case testing: coordinates near tile boundaries
    print(f"\nEdge Case Testing:")
    print("=" * 20)
    
    # Test coordinates very close to tile boundaries
    edge_cases = [
        ("Tile boundary X", -0.0001, 0.0),
        ("Tile boundary Y", 0.0, -0.0001),
        ("Both boundaries", -0.0001, -0.0001),
    ]
    
    zoom = 10
    for name, test_lon, test_lat in edge_cases:
        tile = mercantile.tile(test_lon, test_lat, zoom)
        bbox = mercantile.bounds(tile)
        
        print(f"{name}:")
        print(f"  Input: ({test_lat:.6f}, {test_lon:.6f})")
        print(f"  Tile: {tile}")
        print(f"  Bbox: [{bbox.west:.6f}, {bbox.south:.6f}, {bbox.east:.6f}, {bbox.north:.6f}]")
        
        # Check if point is actually within the tile bounds
        within_bounds = (bbox.west <= test_lon <= bbox.east and 
                        bbox.south <= test_lat <= bbox.north)
        print(f"  Within bounds: {within_bounds}")
    
    print("mercantile example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mercantile example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: mercantile functionality test 4

In [ ]:
import mercantile

# Example 4: Quadkey operations and tile addressing schemes
print("mercantile quadkey operations and tile addressing schemes")

try:
    print("Example 4 - Quadkey operations and tile addressing schemes")
    
    # Define sample tiles for quadkey demonstration
    sample_tiles = [
        mercantile.Tile(x=0, y=0, z=1),      # Top-level tile
        mercantile.Tile(x=1, y=1, z=2),      # Mid-level tile
        mercantile.Tile(x=4823, y=6160, z=14),  # Manhattan area
        mercantile.Tile(x=16962, y=10477, z=16),  # High-resolution tile
    ]
    
    print("Tile to Quadkey Conversions:")
    print("=" * 35)
    
    for tile in sample_tiles:
        # Convert tile to quadkey
        quadkey = mercantile.quadkey(tile)
        
        # Convert quadkey back to tile
        tile_from_qk = mercantile.quadkey_to_tile(quadkey)
        
        print(f"\nTile: {tile}")
        print(f"Quadkey: '{quadkey}'")
        print(f"Round-trip: {tile_from_qk}")
        print(f"Match: {tile == tile_from_qk}")
        
        # Analyze quadkey structure
        print(f"Quadkey analysis:")
        print(f"  Length: {len(quadkey)} (zoom level)")
        print(f"  Path: {' -> '.join(quadkey)}")
        
        # Show tile bounds
        bbox = mercantile.bounds(tile)
        print(f"  Bounds: [{bbox.west:.6f}, {bbox.south:.6f}, {bbox.east:.6f}, {bbox.north:.6f}]")
    
    # Demonstrate quadkey hierarchy and navigation
    print(f"\nQuadkey Hierarchy Navigation:")
    print("=" * 35)
    
    # Start with a mid-level tile
    base_tile = mercantile.Tile(x=1205, y=1540, z=12)  # NYC area
    base_quadkey = mercantile.quadkey(base_tile)
    
    print(f"Base tile: {base_tile}")
    print(f"Base quadkey: '{base_quadkey}'")
    
    # Navigate up the hierarchy (parents)
    print(f"\nParent hierarchy:")
    current_tile = base_tile
    for level in range(3):  # Show 3 parent levels
        parent_tile = mercantile.parent(current_tile)
        parent_quadkey = mercantile.quadkey(parent_tile)
        
        print(f"  Level {parent_tile.z}: {parent_tile} -> '{parent_quadkey}'")
        current_tile = parent_tile
    
    # Navigate down the hierarchy (children)
    print(f"\nChildren of base tile:")
    children = list(mercantile.children(base_tile))
    for i, child in enumerate(children):
        child_quadkey = mercantile.quadkey(child)
        print(f"  Child {i+1}: {child} -> '{child_quadkey}'")
    
    # Quadkey pattern analysis
    print(f"\nQuadkey Pattern Analysis:")
    print("=" * 30)
    
    # Analyze how quadkeys encode spatial relationships
    test_area = [-74.0, 40.7, -73.9, 40.8]  # Manhattan area
    west, south, east, north = test_area
    
    zoom_level = 12
    tiles = list(mercantile.tiles(west, south, east, north, zoom_level))
    
    print(f"Analyzing {len(tiles)} tiles in Manhattan area at zoom {zoom_level}")
    
    # Group tiles by quadkey prefixes
    quadkey_prefixes = {}
    for tile in tiles[:10]:  # Limit to first 10 for readability
        qk = mercantile.quadkey(tile)
        for prefix_len in range(1, len(qk)):
            prefix = qk[:prefix_len]
            if prefix not in quadkey_prefixes:
                quadkey_prefixes[prefix] = []
            quadkey_prefixes[prefix].append(tile)
    
    print(f"Quadkey prefixes found: {len(quadkey_prefixes)}")
    
    # Show some example patterns
    for prefix in sorted(quadkey_prefixes.keys())[:5]:
        tiles_with_prefix = quadkey_prefixes[prefix]
        print(f"  Prefix '{prefix}': {len(tiles_with_prefix)} tiles")
    
    # Quadkey utility functions
    print(f"\nQuadkey Utilities:")
    print("=" * 20)
    
    sample_quadkeys = [
        "0",
        "02",
        "023",
        "0231",
        "023101122"
    ]
    
    for qk in sample_quadkeys:
        tile = mercantile.quadkey_to_tile(qk)
        bbox = mercantile.bounds(tile)
        
        # Calculate tile area in square degrees
        area_sq_deg = (bbox.east - bbox.west) * (bbox.north - bbox.south)
        
        print(f"Quadkey '{qk}':")
        print(f"  Tile: {tile}")
        print(f"  Area: {area_sq_deg:.6f} square degrees")
        print(f"  Zoom level: {len(qk)}")
        
        # Verify quadkey encoding
        reconstructed_qk = mercantile.quadkey(tile)
        print(f"  Verification: {'✓' if qk == reconstructed_qk else '✗'}")
    
    # Performance comparison: XYZ vs Quadkey addressing
    print(f"\nAddressing Scheme Comparison:")
    print("=" * 35)
    
    import time
    
    # Generate test tiles
    test_tiles = []
    for x in range(100, 110):
        for y in range(200, 210):
            test_tiles.append(mercantile.Tile(x=x, y=y, z=10))
    
    # Time XYZ to string conversion
    start_time = time.time()
    xyz_strings = [f"{tile.x}-{tile.y}-{tile.z}" for tile in test_tiles]
    xyz_time = time.time() - start_time
    
    # Time quadkey conversion
    start_time = time.time()
    quadkeys = [mercantile.quadkey(tile) for tile in test_tiles]
    qk_time = time.time() - start_time
    
    print(f"Processed {len(test_tiles)} tiles:")
    print(f"  XYZ addressing: {xyz_time*1000:.2f} ms")
    print(f"  Quadkey addressing: {qk_time*1000:.2f} ms")
    print(f"  Quadkey overhead: {((qk_time/xyz_time)-1)*100:+.1f}%")
    
    # String length comparison
    avg_xyz_len = sum(len(s) for s in xyz_strings) / len(xyz_strings)
    avg_qk_len = sum(len(qk) for qk in quadkeys) / len(quadkeys)
    
    print(f"  Average XYZ string length: {avg_xyz_len:.1f} chars")
    print(f"  Average quadkey length: {avg_qk_len:.1f} chars")
    
    print("mercantile example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mercantile example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: mercantile functionality test 5

In [ ]:
import mercantile
import json

# Example 5: Production workflows and web mapping integration
print("mercantile production workflows and web mapping integration")

try:
    print("Example 5 - Production workflows and web mapping integration")
    
    # Simulate a production tile server workflow
    print("Production Tile Server Workflow Simulation")
    print("=" * 45)
    
    # Define service areas for different map applications
    service_areas = {
        "city_explorer": {
            "name": "City Explorer App",
            "bbox": [-74.05, 40.68, -73.90, 40.88],  # NYC
            "zoom_range": (10, 18),
            "priority": "high"
        },
        "regional_view": {
            "name": "Regional Planning Tool", 
            "bbox": [-74.5, 40.4, -73.5, 41.1],     # Greater NYC
            "zoom_range": (8, 14),
            "priority": "medium"
        },
        "global_overview": {
            "name": "Global Overview Dashboard",
            "bbox": [-180, -85, 180, 85],            # World
            "zoom_range": (0, 6),
            "priority": "low"
        }
    }
    
    # Calculate tile requirements for each service
    total_tiles = 0
    service_stats = {}
    
    for service_id, config in service_areas.items():
        name = config["name"]
        bbox = config["bbox"]
        zoom_range = config["zoom_range"]
        
        print(f"\n📱 {name}")
        print(f"   Area: [{bbox[0]:.2f}, {bbox[1]:.2f}, {bbox[2]:.2f}, {bbox[3]:.2f}]")
        
        service_tiles = 0
        zoom_breakdown = {}
        
        for zoom in range(zoom_range[0], zoom_range[1] + 1):
            tiles = list(mercantile.tiles(bbox[0], bbox[1], bbox[2], bbox[3], zoom))
            tile_count = len(tiles)
            service_tiles += tile_count
            zoom_breakdown[zoom] = tile_count
            
            if zoom <= zoom_range[0] + 2 or zoom >= zoom_range[1] - 1:  # Show first few and last zoom levels
                print(f"   Zoom {zoom:2d}: {tile_count:7,} tiles")
            elif zoom == zoom_range[0] + 3:
                print("   ...      (intermediate levels)")
        
        service_stats[service_id] = {
            'total_tiles': service_tiles,
            'zoom_breakdown': zoom_breakdown,
            'priority': config['priority']
        }
        
        total_tiles += service_tiles
        print(f"   Subtotal: {service_tiles:,} tiles")
    
    print(f"\n💾 Total tiles across all services: {total_tiles:,}")
    
    # Estimate storage and bandwidth requirements
    print(f"\nStorage and Bandwidth Estimates:")
    print("=" * 35)
    
    # Typical tile sizes (KB)
    tile_sizes = {
        'satellite': 25,    # JPEG satellite imagery
        'streets': 15,      # Vector-rendered street map
        'terrain': 20,      # Terrain/topographic
        'labels': 8         # Text labels overlay
    }
    
    for layer_type, size_kb in tile_sizes.items():
        layer_storage_mb = (total_tiles * size_kb) / 1024
        layer_storage_gb = layer_storage_mb / 1024
        
        print(f"{layer_type.title()} layer:")
        print(f"  Storage: {layer_storage_gb:.1f} GB ({size_kb} KB/tile)")
        
        # Estimate daily bandwidth (assume 10% of tiles requested daily)
        daily_requests = total_tiles * 0.1
        daily_bandwidth_gb = (daily_requests * size_kb) / (1024 * 1024)
        print(f"  Daily bandwidth: {daily_bandwidth_gb:.1f} GB (10% cache miss rate)")
    
    # Tile URL generation for different services
    print(f"\nTile URL Generation Examples:")
    print("=" * 35)
    
    # Sample coordinates for URL generation
    sample_location = (-73.9857, 40.7484)  # Times Square, NYC
    lon, lat = sample_location
    
    tile_url_templates = {
        'openstreetmap': 'https://tile.openstreetmap.org/{z}/{x}/{y}.png',
        'google': 'https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        'custom': 'https://tiles.example.com/{layer}/{z}/{x}/{y}.png'
    }
    
    zoom = 15
    tile = mercantile.tile(lon, lat, zoom)
    
    print(f"Sample location: ({lat:.4f}, {lon:.4f}) at zoom {zoom}")
    print(f"Tile coordinates: {tile}")
    print(f"\nGenerated URLs:")
    
    for service, template in tile_url_templates.items():
        if service == 'custom':
            url = template.format(
                layer='streets',
                z=tile.z, x=tile.x, y=tile.y
            )
        else:
            url = template.format(z=tile.z, x=tile.x, y=tile.y)
        
        print(f"  {service.title()}: {url}")
    
    # Generate TileJSON specification
    print(f"\nTileJSON Specification Generation:")
    print("=" * 40)
    
    # Create TileJSON metadata for a sample service
    tilejson = {
        "tilejson": "3.0.0",
        "name": "City Explorer Tiles",
        "description": "High-resolution city mapping tiles",
        "version": "1.0.0",
        "attribution": "© OpenStreetMap contributors",
        "scheme": "xyz",
        "tiles": [
            "https://tiles.example.com/city/{z}/{x}/{y}.png"
        ],
        "minzoom": service_areas["city_explorer"]["zoom_range"][0],
        "maxzoom": service_areas["city_explorer"]["zoom_range"][1],
        "bounds": service_areas["city_explorer"]["bbox"],
        "center": [
            (service_areas["city_explorer"]["bbox"][0] + service_areas["city_explorer"]["bbox"][2]) / 2,
            (service_areas["city_explorer"]["bbox"][1] + service_areas["city_explorer"]["bbox"][3]) / 2,
            12  # Default zoom level
        ]
    }
    
    print("Generated TileJSON specification:")
    print(json.dumps(tilejson, indent=2)[:500] + "...")
    
    # Performance optimization recommendations
    print(f"\nPerformance Optimization Recommendations:")
    print("=" * 50)
    
    high_priority_services = [k for k, v in service_stats.items() if v['priority'] == 'high']
    
    print(f"High-priority services: {len(high_priority_services)}")
    
    for service_id in high_priority_services:
        stats = service_stats[service_id]
        config = service_areas[service_id]
        
        print(f"\n🚀 {config['name']}:")
        print(f"   Total tiles: {stats['total_tiles']:,}")
        
        # Recommend pre-generation for smaller tile counts
        if stats['total_tiles'] < 100000:
            print(f"   ✓ Recommend: Pre-generate all tiles")
        else:
            print(f"   ✓ Recommend: On-demand generation with aggressive caching")
        
        # Analyze zoom level distribution
        max_tiles_zoom = max(stats['zoom_breakdown'].items(), key=lambda x: x[1])
        print(f"   Peak zoom level: {max_tiles_zoom[0]} ({max_tiles_zoom[1]:,} tiles)")
        
        # CDN recommendations
        bbox = config['bbox']
        bbox_width = bbox[2] - bbox[0]
        bbox_height = bbox[3] - bbox[1]
        
        if bbox_width < 1 and bbox_height < 1:  # Very localized
            print(f"   ✓ Recommend: Single-region CDN deployment")
        else:
            print(f"   ✓ Recommend: Multi-region CDN with edge caching")
    
    # Integration code example
    print(f"\nIntegration Code Example (Python):")
    print("=" * 40)
    
    integration_code = '''
def get_tiles_for_viewport(bbox, zoom):
    """Get all tiles needed for a map viewport"""
    west, south, east, north = bbox
    return list(mercantile.tiles(west, south, east, north, zoom))

def generate_tile_urls(tiles, base_url_template):
    """Generate URLs for a list of tiles"""
    return [
        base_url_template.format(z=tile.z, x=tile.x, y=tile.y)
        for tile in tiles
    ]

# Usage example:
viewport = [-74.01, 40.70, -73.97, 40.75]  # Small NYC area
zoom_level = 16
tiles_needed = get_tiles_for_viewport(viewport, zoom_level)
urls = generate_tile_urls(tiles_needed, "https://tiles.example.com/{z}/{x}/{y}.png")
'''
    
    print(integration_code.strip())
    
    print(f"\nWorkflow Summary:")
    print(f"  Services analyzed: {len(service_areas)}")
    print(f"  Total tile calculations: {sum(len(stats['zoom_breakdown']) for stats in service_stats.values())}")
    print(f"  URL templates generated: {len(tile_url_templates)}")
    print(f"  TileJSON specification: ✓ Created")
    
    print("mercantile example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"mercantile example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")
print(f"\nMercantile capabilities demonstrated:")
print("- Basic tile operations and coordinate transformations")
print("- Bounding box operations and tile range calculations")
print("- Projection transformations and coordinate system handling")
print("- Quadkey operations and alternative tile addressing schemes")
print("- Production workflows and web mapping service integration")